In [ ]:
import torch

# ---- Model Definition ----
class AttentionGraph(torch.nn.Module):
    def __init__(self, B=1, M=8192, H=96, E=128):
        super().__init__()

        D = H * E
        F = E
        G = H * E
        C = 4 * H * E
        J = H * E

        # Register weights as parameters
        self.WV = torch.nn.Parameter(torch.randn(H, E, D))
        self.WK = torch.nn.Parameter(torch.randn(H, E, D))
        self.WQ = torch.nn.Parameter(torch.randn(H, E, D))
        self.WV = torch.nn.Parameter(torch.randn(H, F, G))
        self.WFFA = torch.nn.Parameter(torch.randn(G, C))
        self.WFFB = torch.nn.Parameter(torch.randn(C, J))

    def forward(self, I_in: torch.Tensor):
        # I copy
        I = I_in

        # Projections
        V = torch.einsum("bmd,hed->bmhe", I, self.WV)
        K = torch.einsum("bmd,hed->bmhe", I, self.WK)
        Q = torch.einsum("bmd,hed->bmhe", I, self.WQ)

        # QK (note: p dimension = m dimension reused)
        QK = torch.einsum("bmhe,bphe->bmph", Q, K)

        # Softmax over p dimension
        QK_softmax = torch.softmax(QK, dim=2)

        # Attention value
        AV = torch.einsum("bmph,bphe->bmhe", QK_softmax, V)

        # Output projection
        V = torch.einsum("bmhf,hfg->bmg", AV, self.WV)

        # Feedforward
        FFA = torch.einsum("bmg,gc->bmc", V, self.WFFA)
        FFB = torch.einsum("bmc,cj->bmj", FFA, self.WFFB)

        return FFB

In [ ]:
model = AttentionGraph(B=1, M=128, H=8, E=32)

scripted_model = torch.jit.script(model)

B, M, H, E = 1, 128, 8, 32
D = H * E

I_in = torch.randn(B, M, D)

output = scripted_model(I_in)
print(output.shape)

In [ ]:
print(scripted_model.inlined_graph)

In [ ]:
with torch.profiler.profile(record_shapes=True) as prof:
    scripted_model(I_in)

print(prof.key_averages().table(sort_by="cpu_time_total"))